# Preprocessing Pipeline — DeepLabCut vs TopScan

Prepares the raw tracking data before the comparative analysis: converts the videos, crops them per animal, exports DeepLabCut coordinates (`.h5` → `.txt`) with a confidence filter, and organizes the raw TopScan files (mosaic offset + exploration events merged).

Must be run **before** the `Analysis Pipeline` notebook. TopScan leaves this notebook still **raw** — without outlier filtering, smoothing or spatial calibration; those steps run once, in the Analysis notebook, applied symmetrically to TopScan and DeepLabCut.

**Flow:**
```
.MPG videos → .MP4 conversion → Per-animal cropping
.H5 files (DLC) → Confidence filter → TXT per bodypart
Raw TopScan TXT → Offset correction → Event merging → Organized TXT
```

Run the sections **in order**. Each is independent after mounting Drive — skip the ones you do not need (e.g., if your videos are already converted and cropped, go straight to the DeepLabCut export).

## Connect to Google Drive

This notebook expects the following structure under `MyDrive/validacao_topscan/`:

```
validacao_topscan/
├── VIDEOS/
│   ├── originais_mpg/        ← raw multi-animal videos (.mpg)
│   ├── convertidos_mp4/      ← conversion output (.mp4)
│   └── recortados/           ← per-animal crop output
├── DEEPLABCUT/
│   ├── h5_exportado/         ← .h5 files exported by DLC
│   └── trajetoria_txt_dlc/   ← TXTs generated by this notebook
└── TOPSCAN/
    ├── TXT original e XLSX original/  ← raw TopScan pairs (.TXT + .xlsx)
    └── TXT novo/                      ← TXTs organized by this notebook
```

> 💡 Adjust the paths in each section's cell if your structure is different.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## (Optional) Install from GitHub instead of Drive

Only run this cell if you prefer to clone the repository for the scripts/example data, instead of mounting your own Drive above.

In [ ]:
# !git clone -q https://github.com/RodrigoOrvate/Crosstrack-validator.git
# %cd Crosstrack-validator

## Install dependencies

In [ ]:
!apt-get install -q ffmpeg
!pip install -q ffmpeg-python moviepy tqdm
!pip install -q --upgrade pandas scikit-learn scipy opencv-python-headless

## Imports and paths

In [ ]:
import glob
import os
import re
import subprocess

import cv2
import ffmpeg
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
RAIZ_DRIVE = "/content/drive/MyDrive/validacao_topscan"

## 1. Video conversion (.MPG → .MP4)

In [ ]:
def converter_videos(pasta_entrada, pasta_saida, prefixo_arquivo="M"):
    os.makedirs(pasta_saida, exist_ok=True)
    arquivos = [f for f in os.listdir(pasta_entrada)
                if f.lower().endswith(".mpg") and f.startswith(prefixo_arquivo)]

    for nome_arquivo in tqdm(arquivos, desc="Converting videos", unit="video"):
        caminho_entrada = os.path.join(pasta_entrada, nome_arquivo)
        caminho_saida = os.path.join(pasta_saida, os.path.splitext(nome_arquivo)[0] + ".mp4")

        try:
            duracao = float(ffmpeg.probe(caminho_entrada)["format"]["duration"])
        except ffmpeg.Error as e:
            print(f"Error reading {nome_arquivo}: {e.stderr.decode('utf-8')}")
            continue

        comando = ["ffmpeg", "-i", caminho_entrada, "-r", "30", caminho_saida]
        processo = subprocess.Popen(comando, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
        while True:
            linha = processo.stderr.readline()
            if not linha and processo.poll() is not None:
                break
        processo.wait()

    print("Conversion complete.")

## 2. Multi-animal video cropping

In [ ]:
def recortar_video_por_animal(caminho_video, pasta_saida, largura_barra=35):
    nome_arquivo = os.path.basename(caminho_video)
    nome_sem_extensao = os.path.splitext(nome_arquivo)[0]

    try:
        prefixo, resto = nome_sem_extensao.split("--", 1)
        ids = [f"{prefixo}-{parte}" for parte in resto.split("-")]
    except ValueError:
        print(f"Skipped file with unexpected name: {nome_arquivo}")
        return

    cap = cv2.VideoCapture(caminho_video)
    if not cap.isOpened():
        print(f"Error opening: {caminho_video}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    codec = cv2.VideoWriter_fourcc(*"mp4v")

    # Quadrants: 1=top-left, 2=top-right, 3=bottom-left
    quadrantes = [
        (35, 0, largura // 2 + 5, altura // 2),
        (largura // 2 + 30, 0, largura - 20, altura // 2),
        (35, altura // 2 + 5, largura // 2 + 5, altura - 10),
    ]

    escritores = {}
    for idx in range(min(len(ids), len(quadrantes))):
        x1, y1, x2, y2 = quadrantes[idx]
        caminho_saida = os.path.join(pasta_saida, f"{ids[idx].strip()}.mp4")
        escritores[idx] = cv2.VideoWriter(caminho_saida, codec, fps, (x2 - x1, y2 - y1))

    cor_barra = (57, 60, 60)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        for idx, escritor in escritores.items():
            x1, y1, x2, y2 = quadrantes[idx]
            recorte = frame[y1:y2, x1:x2].copy()
            recorte[:, -largura_barra:] = cor_barra  # hides the timestamp overlaid by TopScan
            escritor.write(recorte)

    cap.release()
    for escritor in escritores.values():
        escritor.release()
    print(f"Done: {nome_arquivo} -> {', '.join(ids[:len(escritores)])}")

## 3. DeepLabCut export (.h5 → .txt)

In [ ]:
def extrair_id_animal(nome_arquivo):
    match = re.search(r"^(.*?)DLC", nome_arquivo, re.IGNORECASE)
    return match.group(1).strip("_") if match else "X"

In [ ]:
def exportar_deeplabcut_para_txt(pasta_h5, pasta_saida, bodypart, limiar_likelihood=0.8):
    """Extracts a bodypart's X/Y from each filtered DeepLabCut .h5 file,
    discards frames with likelihood below the threshold (turned into NaN),
    and writes one .txt per animal. Automatically separates held-out
    (leave-one-mosaic-out) models from full models by the file name DLC
    exported."""
    os.makedirs(pasta_saida, exist_ok=True)
    caminhos_h5 = glob.glob(os.path.join(pasta_h5, "*filtered.h5"))
    relatorio_qualidade = []

    for caminho_h5 in caminhos_h5:
        arquivo = os.path.basename(caminho_h5)
        rat_id = extrair_id_animal(arquivo)

        if "networkheldout" in arquivo.lower():
            match_mosaico = re.search(r"networkheldout(m\d)", arquivo, re.IGNORECASE)
            mosaico = match_mosaico.group(1).upper() if match_mosaico else "MX"
            pasta_destino = os.path.join(pasta_saida, "holdouts", mosaico)
            sufixo = f"heldout_{mosaico}"
        else:
            pasta_destino = os.path.join(pasta_saida, "full")
            sufixo = "full"
        os.makedirs(pasta_destino, exist_ok=True)

        try:
            df = pd.read_hdf(caminho_h5)
            scorer = df.columns.get_level_values(0)[0]
            dados_bodypart = df[scorer][bodypart]

            x, y, likelihood = dados_bodypart["x"], dados_bodypart["y"], dados_bodypart["likelihood"]
            mascara_baixa_confianca = likelihood < limiar_likelihood
            x = x.where(~mascara_baixa_confianca, other=np.nan)
            y = y.where(~mascara_baixa_confianca, other=np.nan)

            nome_txt = f"ID_{rat_id}_trajetoria_{bodypart}_{sufixo}.txt"
            pd.DataFrame({"Frame": range(len(df)), "X": x.round(2), "Y": y.round(2)}).to_csv(
                os.path.join(pasta_destino, nome_txt), sep=" ", index=False, float_format="%.2f", na_rep="NaN"
            )
            print(f"{rat_id} -> {pasta_destino}")

            relatorio_qualidade.append({
                "Animal_ID": rat_id,
                "Categoria": sufixo,
                "Arquivo": arquivo,
                "Total_Frames": len(df),
                "Frames_Rejeitados": int(mascara_baixa_confianca.sum()),
                "Perda_Percentual": round(mascara_baixa_confianca.sum() / len(df) * 100, 2),
            })
        except KeyError:
            print(f"Bodypart '{bodypart}' missing in: {arquivo}")
        except Exception as e:
            print(f"Failure in {arquivo}: {e}")

    caminho_relatorio = os.path.join(pasta_saida, "relatorio_qualidade_dlc.csv")
    pd.DataFrame(relatorio_qualidade).to_csv(caminho_relatorio, index=False)
    print(f"Quality report saved to: {caminho_relatorio}")

## 4. Raw TopScan organization

In [ ]:
def _ler_eventos(eventos_path):
    """Reads the events table (From Frame, To Frame, Event) from .xlsx
    (original TopScan format, header on the 2nd row) or .csv."""
    if eventos_path.lower().endswith(".csv"):
        df = pd.read_csv(eventos_path, header=0)
    else:
        df = pd.read_excel(eventos_path, header=1)
    df.columns = df.columns.str.strip()
    return df

In [ ]:
def organizar_topscan(txt_path, eventos_path, saida_path):
    print(f"Processing {os.path.basename(txt_path)}...")

    try:
        colunas = ["FrameNum", "CenterX(mm)", "CenterY(mm)", "Areas"]
        df = pd.read_csv(txt_path, sep=r"\s+", skiprows=1, names=colunas)
        df["Areas"] = df["Areas"].astype(str).str.replace(",", "", regex=False)
    except Exception as e:
        print(f"  Error reading the TopScan TXT: {e}")
        return

    df["CenterX(mm)"] = pd.to_numeric(df["CenterX(mm)"], errors="coerce")
    df["CenterY(mm)"] = pd.to_numeric(df["CenterY(mm)"], errors="coerce")
    df = df[(df["CenterX(mm)"] > 0) & (df["CenterY(mm)"] > 0)].copy()

    # Each recording contains up to 3 side-by-side arenas (mosaic); corrects
    # the offset according to the animal's quadrant, identified from the file
    # name (e.g., "AA-BB-CC_2_..." -> second animal in the mosaic, top-right
    # quadrant).
    nome_base = os.path.splitext(os.path.basename(txt_path))[0]
    match = re.search(r"([A-Za-z\-]+)_(\d+)", nome_base)
    indice = int(match.group(2)) - 1 if match else 0
    largura_campo, altura_campo = 720, 480
    offset_x, offset_y = {0: (0, 0), 1: (largura_campo, 0), 2: (0, altura_campo)}.get(indice, (0, 0))
    df["CenterX(mm)"] -= offset_x
    df["CenterY(mm)"] -= offset_y

    try:
        df_eventos = _ler_eventos(eventos_path)
        df["Areas"] = "Floor"
        for _, row in df_eventos.iterrows():
            evento = str(row["Event"])
            evento = evento.split("sniffing On")[-1].strip() if "sniffing On" in evento else evento.strip()
            df.loc[(df["FrameNum"] >= int(row["From Frame"])) & (df["FrameNum"] <= int(row["To Frame"])), "Areas"] = evento
    except Exception as e:
        print(f"  Error processing events: {e}")
        return

    os.makedirs(os.path.dirname(saida_path), exist_ok=True)
    with open(saida_path, "w") as f_out:
        f_out.write("FrameNum CenterX(mm) CenterY(mm) Areas\n")
        df.to_csv(f_out, sep=" ", index=False, header=False, float_format="%.2f", na_rep="NaN")
    print(f"  Saved: {os.path.basename(saida_path)}")

In [ ]:
def organizar_topscan_em_lote(pasta_entrada, pasta_saida):
    if not os.path.isdir(pasta_entrada):
        print(f"Folder not found: {pasta_entrada}")
        return
    for nome_txt in sorted(f for f in os.listdir(pasta_entrada) if f.upper().endswith(".TXT")):
        nome_base = os.path.splitext(nome_txt)[0]
        txt_path = os.path.join(pasta_entrada, nome_txt)
        xlsx_path = os.path.join(pasta_entrada, f"{nome_base}.xlsx")
        csv_path = os.path.join(pasta_entrada, f"{nome_base}_eventos.csv")
        saida_path = os.path.join(pasta_saida, f"{nome_base}_NOVO.TXT")

        if os.path.exists(csv_path):
            organizar_topscan(txt_path, csv_path, saida_path)
        elif os.path.exists(xlsx_path):
            organizar_topscan(txt_path, xlsx_path, saida_path)
        else:
            print(f"Events missing for {nome_txt} -- skipped.")

### Run Section 1 (video conversion)

In [ ]:
converter_videos(
    pasta_entrada=os.path.join(RAIZ_DRIVE, "VIDEOS", "originais_mpg"),
    pasta_saida=os.path.join(RAIZ_DRIVE, "VIDEOS", "convertidos_mp4"),
)

### Run Section 2 (per-animal cropping)

In [ ]:
pasta_videos = os.path.join(RAIZ_DRIVE, "VIDEOS", "convertidos_mp4")
pasta_recortados = os.path.join(RAIZ_DRIVE, "VIDEOS", "recortados")
os.makedirs(pasta_recortados, exist_ok=True)

for arquivo in os.listdir(pasta_videos):
    if arquivo.endswith(".mp4") and "--" in arquivo:
        recortar_video_por_animal(os.path.join(pasta_videos, arquivo), pasta_recortados)

### Run Section 3 (DeepLabCut export)

In [ ]:
exportar_deeplabcut_para_txt(
    pasta_h5=os.path.join(RAIZ_DRIVE, "DEEPLABCUT", "h5_exportado"),
    pasta_saida=os.path.join(RAIZ_DRIVE, "DEEPLABCUT", "trajetoria_txt_dlc"),
    bodypart="body",
    limiar_likelihood=0.8,
)

### Run Section 4 (TopScan organization)

In [ ]:
organizar_topscan_em_lote(
    pasta_entrada=os.path.join(RAIZ_DRIVE, "TOPSCAN", "TXT original e XLSX original"),
    pasta_saida=os.path.join(RAIZ_DRIVE, "TOPSCAN", "TXT novo"),
)

---
### Note on a lab-specific step

The original version of this pipeline included an additional TopScan spatial-calibration step by regression (against a DeepLabCut reference), used for a lab-specific pilot dataset. That step is no longer part of the current pipeline: spatial calibration now runs once, symmetrically, in the Analysis notebook. Nothing beyond the 4 sections above needs to be run.